# 📌 Parámetros para optimizar un modelo KNN


## 1 - IMPORTAMOS LIBRERIAS MAS COMUNES

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

## 2 - CARGAMOS DATASET

In [2]:
penguins_df = sns.load_dataset('penguins').dropna()
penguins_df

,species,island,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g,sex
0,Adelie,Torgersen,39.1,18.7,181.0,3750.0,Male
1,Adelie,Torgersen,39.5,17.4,186.0,3800.0,Female
2,Adelie,Torgersen,40.3,18.0,195.0,3250.0,Female
4,Adelie,Torgersen,36.7,19.3,193.0,3450.0,Female
5,Adelie,Torgersen,39.3,20.6,190.0,3650.0,Male
...,...,...,...,...,...,...,...
338,Gentoo,Biscoe,47.2,13.7,214.0,4925.0,Female
340,Gentoo,Biscoe,46.8,14.3,215.0,4850.0,Female
341,Gentoo,Biscoe,50.4,15.7,222.0,5750.0,Male
342,Gentoo,Biscoe,45.2,14.8,212.0,5200.0,Female


## 3 - EDA

In [3]:
penguins_df.shape

(333, 7)

In [4]:
penguins_df.dtypes

,0
species,object
island,object
bill_length_mm,float64
bill_depth_mm,float64
flipper_length_mm,float64
body_mass_g,float64
sex,object


### 3.1 REVISAMOS LAS DIFERENTES ESPECIES DE PINGUINOS PARA PODER CREAR UN MODELO DE CLASIFICACIÓN

In [5]:
penguins_df['species'].value_counts()

,count
species,
Adelie,146
Gentoo,119
Chinstrap,68


### 3.2 CODIFICACIÓN DE VARIABLES CATEGORICA SPECIES

In [6]:
penguins_df.loc[:,'species'] = penguins_df['species'].map({'Adelie':0,'Gentoo':1,'Chinstrap':2})
penguins_df.loc[:,'sex'] = penguins_df['sex'].map({'Male':0,'Female':1})
penguins_df['species'] = penguins_df['species'].astype(int)
penguins_df['sex'] = penguins_df['sex'].astype(int)


In [7]:
penguins_df.dtypes

,0
species,int64
island,object
bill_length_mm,float64
bill_depth_mm,float64
flipper_length_mm,float64
body_mass_g,float64
sex,int64


# 4 - PARAMETROS MAS COMUNES

# Parámetros para mejorar el modelo KNeighborsClassifier

El rendimiento del modelo **K-Nearest Neighbors (KNN)** puede mejorarse ajustando distintos hiperparámetros:

## 4.1 IDENTIFICAMOS VARIABLES X y Y

In [8]:
X = penguins_df[['bill_length_mm', 'bill_depth_mm', 'flipper_length_mm', 'body_mass_g', 'sex']]  # Seleccionar características
y = penguins_df['species']

## 4.2 IMPORTAMOS LIBRERIAS

In [9]:
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report,accuracy_score

## 4.3 DIVIDIMOS EL DATASET EN ENTRENAMIENTO Y PRUEBA

In [10]:
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=42)

## 4.4 ESCALAMOS LOS DATOS

In [11]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.fit_transform(X_test)

## 4.5 PARAMETROS

## 1 Número de vecinos (`n_neighbors`)
- Controla el número de vecinos que se consideran para la clasificación.
- Valores pequeños pueden llevar a **sobreajuste**, mientras que valores grandes pueden provocar **subajuste**.
- Se recomienda probar distintos valores y seleccionar el óptimo con **validación cruzada**.

In [12]:
knn = KNeighborsClassifier(n_neighbors=3)
knn.fit(X_train_scaled,y_train)
y_pred = knn.predict(X_test_scaled)
accuracy = accuracy_score(y_test, y_pred)
print(f'Accuracy: {accuracy:.2f}')

Accuracy: 0.99


## 2. Peso de los vecinos (`weights`)
Define cómo contribuyen los vecinos a la clasificación:
- `"uniform"` (predeterminado): todos los vecinos tienen el mismo peso.
- `"distance"`: los vecinos más cercanos tienen mayor peso en la predicción.
- También se puede proporcionar una función personalizada.

In [13]:
knn = KNeighborsClassifier(n_neighbors=5,weights='distance')
knn.fit(X_train_scaled,y_train)
y_pred = knn.predict(X_test_scaled)
accuracy = accuracy_score(y_test, y_pred)
print(f'Accuracy: {accuracy:.2f}')

Accuracy: 1.00


## 3. Método de cálculo de distancia (`metric`)
- `"minkowski"` (predeterminado): se usa la distancia Euclidiana (`p=2`) o Manhattan (`p=1`).
- `"euclidean"`: distancia Euclidiana.
- `"manhattan"`: distancia basada en la suma de diferencias absolutas.
- `"chebyshev"`: máxima diferencia entre dimensiones.
- `"mahalanobis"`: considera la correlación entre variables.

In [14]:
knn = KNeighborsClassifier(n_neighbors=5,weights='distance',metric='manhattan')
knn.fit(X_train_scaled,y_train)
y_pred = knn.predict(X_test_scaled)
accuracy = accuracy_score(y_test, y_pred)
print(f'Accuracy: {accuracy:.2f}')

Accuracy: 0.97


## 4. Algoritmo de búsqueda (`algorithm`)
Controla cómo se encuentran los vecinos más cercanos:
- `"auto"`: selecciona el mejor algoritmo según los datos.
- `"ball_tree"`: útil para grandes conjuntos de datos.
- `"kd_tree"`: eficiente en datos de baja dimensión.
- `"brute"`: búsqueda fuerza bruta, útil cuando los datos son pequeños.

In [15]:
knn = KNeighborsClassifier(n_neighbors=5,weights='distance',metric='manhattan',algorithm='brute')
knn.fit(X_train_scaled,y_train)
y_pred = knn.predict(X_test_scaled)
accuracy = accuracy_score(y_test, y_pred)
print(f'Accuracy: {accuracy:.2f}')

Accuracy: 0.97


## 5. Número de dimensiones (`leaf_size`)
- Aplica solo a los algoritmos `"ball_tree"` y `"kd_tree"`.
- Un menor `leaf_size` mejora la precisión, pero aumenta el tiempo de cómputo.

In [16]:
knn = KNeighborsClassifier(n_neighbors=5,weights='distance',metric='manhattan',algorithm='kd_tree',leaf_size=10)
knn.fit(X_train_scaled,y_train)
y_pred = knn.predict(X_test_scaled)
accuracy = accuracy_score(y_test, y_pred)
print(f'Accuracy: {accuracy:.2f}')

Accuracy: 0.97


## 6. Parámetro `p` en la métrica Minkowski
- `p=1` → Distancia Manhattan.
- `p=2` → Distancia Euclidiana.
- Valores mayores de `p` cambian la forma en que se mide la distancia entre puntos.

In [17]:
knn = KNeighborsClassifier(n_neighbors=5,weights='distance',metric='minkowski',algorithm='kd_tree',leaf_size=10,p=2)
knn.fit(X_train_scaled,y_train)
y_pred = knn.predict(X_test_scaled)
accuracy = accuracy_score(y_test, y_pred)
print(f'Accuracy: {accuracy:.2f}')

Accuracy: 1.00
